In [2]:
from pathlib import Path
import re
import pandas as pd

# Notebook is in: notebook/external_test.ipynb
# data/ is sibling of notebook/
PROJECT_ROOT = Path("..").resolve()
ASSEMBLIES_ROOT = PROJECT_ROOT / "data" / "assemblies"

def read_text(p: Path) -> str:
    try:
        return p.read_text(errors="ignore")
    except Exception:
        return ""

def parse_getorg_log(log_path: Path):
    """
    Pull key metrics from get_org.log.txt
    """
    txt = read_text(log_path)

    # Reads used = 10000+10000 (pairs ~= first number)
    m = re.search(r"Reads used\s*=\s*(\d+)\+(\d+)", txt)
    reads_r1 = int(m.group(1)) if m else None
    reads_r2 = int(m.group(2)) if m else None
    reads_pairs = reads_r1

    # Result status of animal_mt: 2 scaffold(s)
    m = re.search(r"Result status of animal_mt:\s*(\d+)\s*scaffold", txt)
    scaffolds = int(m.group(1)) if m else None

    # Average animal_mt base-coverage = 114.9
    m = re.search(r"Average animal_mt base-coverage\s*=\s*([0-9.]+)", txt)
    avg_base_cov = float(m.group(1)) if m else None

    # Sometimes: Average animal_mt coverage = 18.6
    m = re.search(r"Average animal_mt coverage\s*=\s*([0-9.]+)", txt)
    avg_cov = float(m.group(1)) if m else None

    return {
        "reads_pairs_used": reads_pairs,
        "reads_r2_used": reads_r2,
        "scaffolds": scaffolds,
        "avg_base_cov": avg_base_cov,
        "avg_cov": avg_cov,
    }

def fasta_stats(fa_path: Path):
    """
    Returns (n_seqs, total_bp) from FASTA
    """
    n = 0
    total = 0
    seq_len = 0
    for line in fa_path.read_text(errors="ignore").splitlines():
        if line.startswith(">"):
            if seq_len > 0:
                total += seq_len
                seq_len = 0
            n += 1
        else:
            seq_len += len(line.strip())
    total += seq_len
    return n, total

def gfa_stats(gfa_path: Path):
    """
    Returns (nodes, edges) from GFA:
      nodes = number of S lines
      edges = number of L lines
    """
    nodes = 0
    edges = 0
    for line in gfa_path.read_text(errors="ignore").splitlines():
        if line.startswith("S\t"):
            nodes += 1
        elif line.startswith("L\t"):
            edges += 1
    return nodes, edges

def find_latest_k_file(run_root: Path, glob_pat: str):
    """
    Returns the file with the largest K in its name (e.g., K115).
    """
    hits = list(run_root.glob(glob_pat))
    if not hits:
        return None

    def k_value(p: Path):
        m = re.search(r"\.K(\d+)\.", p.name)
        return int(m.group(1)) if m else -1

    hits.sort(key=k_value, reverse=True)
    return hits[0]

def summarize_run(run_root: Path):
    """
    run_root: .../getorganelle_unfiltered or .../getorganelle_filtered_<tag>
    """
    log_path = run_root / "get_org.log.txt"
    if not log_path.exists():
        return None

    logm = parse_getorg_log(log_path)

    # Prefer largest K outputs
    fa = find_latest_k_file(run_root, "animal_mt.K*.scaffolds.graph1.1.path_sequence.fasta")
    if fa is None:
        fa = find_latest_k_file(run_root, "animal_mt.K*.scaffolds*.fasta")

    gfa = find_latest_k_file(run_root, "animal_mt.K*.contigs.graph1.selected_graph.gfa")
    if gfa is None:
        gfa = find_latest_k_file(run_root, "animal_mt.K*.contigs*.gfa")

    nseqs = total_bp = None
    if fa and fa.exists():
        nseqs, total_bp = fasta_stats(fa)

    nodes = edges = None
    if gfa and gfa.exists():
        nodes, edges = gfa_stats(gfa)

    return {
        "run_root": str(run_root),
        **logm,
        "assembly_fasta": str(fa) if fa else None,
        "fasta_seqs": nseqs,
        "assembly_bp": total_bp,
        "graph_gfa": str(gfa) if gfa else None,
        "gfa_nodes": nodes,
        "gfa_edges": edges,
    }

print("PROJECT_ROOT =", PROJECT_ROOT)
print("ASSEMBLIES_ROOT =", ASSEMBLIES_ROOT)

PROJECT_ROOT = /Users/yvonnelin/Desktop/mitochime
ASSEMBLIES_ROOT = /Users/yvonnelin/Desktop/mitochime/data/assemblies


In [9]:
# Datasets you want
DATASETS = [
    "10K_ext5_mixed",
    "10K_ext10_mixed",
    "10K_ext20_mixed",
    "10K_ext50_mixed",
]

FILTERS = ["unfiltered", "gb", "cnn", "trf"]

rows = []

for ds in DATASETS:
    for flt in FILTERS:
        if flt == "unfiltered":
            folder = f"{ds}_unfiltered_cli"
            run_root = ASSEMBLIES_ROOT / folder / "getorganelle_unfiltered"
        else:
            folder = f"{ds}_{flt}_filtered_cli"
            run_root = ASSEMBLIES_ROOT / folder / f"getorganelle_filtered_{flt}"

        rec = summarize_run(run_root)
        if rec is None:
            rows.append({
                "dataset": ds,
                "filter": flt,
                "status": "MISSING",
                "run_root": str(run_root),
            })
        else:
            rows.append({
                "dataset": ds,
                "filter": flt,
                "status": "OK",
                **rec,
            })

df = pd.DataFrame(rows).sort_values(["dataset","filter"]).reset_index(drop=True)

# Show missing runs (if any)
missing = df[df["status"] != "OK"][["dataset","filter","run_root"]]
if len(missing):
    print("Missing runs (folder or log not found):")
    display(missing)

df

,dataset,filter,status,run_root,reads_pairs_used,reads_r2_used,scaffolds,avg_base_cov,avg_cov,assembly_fasta,fasta_seqs,assembly_bp,graph_gfa,gfa_nodes,gfa_edges
0,10K_ext10_mixed,cnn,OK,/Users/yvonnelin/Desktop/mitochime/data/assemb...,7734,7734,1,124.3,29.8,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,16612,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,0
1,10K_ext10_mixed,gb,OK,/Users/yvonnelin/Desktop/mitochime/data/assemb...,9547,9547,1,143.1,34.3,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,16612,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,0
2,10K_ext10_mixed,trf,OK,/Users/yvonnelin/Desktop/mitochime/data/assemb...,8189,8189,1,130.9,31.4,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,16612,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,0
3,10K_ext10_mixed,unfiltered,OK,/Users/yvonnelin/Desktop/mitochime/data/assemb...,10000,10000,1,143.8,34.5,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,16612,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,0
4,10K_ext20_mixed,cnn,OK,/Users/yvonnelin/Desktop/mitochime/data/assemb...,7059,7059,1,111.2,26.7,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,16615,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,0
5,10K_ext20_mixed,gb,OK,/Users/yvonnelin/Desktop/mitochime/data/assemb...,9126,9126,1,134.0,32.1,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,16597,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,0
6,10K_ext20_mixed,trf,OK,/Users/yvonnelin/Desktop/mitochime/data/assemb...,7561,7561,1,117.2,28.1,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,16615,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,0
7,10K_ext20_mixed,unfiltered,OK,/Users/yvonnelin/Desktop/mitochime/data/assemb...,10000,10000,1,134.7,32.3,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,16669,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,0
8,10K_ext50_mixed,cnn,OK,/Users/yvonnelin/Desktop/mitochime/data/assemb...,4857,4857,1,70.6,16.9,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,16543,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,0
9,10K_ext50_mixed,gb,OK,/Users/yvonnelin/Desktop/mitochime/data/assemb...,7975,7975,2,109.3,NaN,/Users/yvonnelin/Desktop/mitochime/data/assemb...,2,16763,/Users/yvonnelin/Desktop/mitochime/data/assemb...,2,0


In [10]:
unf = df[(df["filter"] == "unfiltered") & (df["status"] == "OK")].copy()
flt = df[(df["filter"] != "unfiltered") & (df["status"] == "OK")].copy()

merged = flt.merge(
    unf[[
        "dataset",
        "reads_pairs_used",
        "scaffolds",
        "assembly_bp",
        "fasta_seqs",
        "gfa_nodes",
        "gfa_edges",
        "avg_base_cov",
    ]],
    on="dataset",
    how="left",
    suffixes=("", "_unf"),
)

merged["reads_kept_pct"] = (merged["reads_pairs_used"] / merged["reads_pairs_used_unf"] * 100).round(2)

results = merged[[
    "dataset","filter",
    "reads_pairs_used_unf","reads_pairs_used","reads_kept_pct",
    "scaffolds_unf","scaffolds",
    "assembly_bp_unf","assembly_bp",
    "fasta_seqs_unf","fasta_seqs",
    "gfa_nodes_unf","gfa_nodes",
    "gfa_edges_unf","gfa_edges",
    "avg_base_cov_unf","avg_base_cov",
    "run_root"
]].rename(columns={
    "reads_pairs_used_unf": "reads_pairs_unfiltered",
    "reads_pairs_used": "reads_pairs_filtered",
    "scaffolds_unf": "scaffolds_unfiltered",
    "scaffolds": "scaffolds_filtered",
    "assembly_bp_unf": "bp_unfiltered",
    "assembly_bp": "bp_filtered",
    "fasta_seqs_unf": "seqs_unfiltered",
    "fasta_seqs": "seqs_filtered",
    "gfa_nodes_unf": "nodes_unfiltered",
    "gfa_nodes": "nodes_filtered",
    "gfa_edges_unf": "edges_unfiltered",
    "gfa_edges": "edges_filtered",
    "avg_base_cov_unf": "avg_base_cov_unfiltered",
    "avg_base_cov": "avg_base_cov_filtered",
}).sort_values(["dataset","filter"]).reset_index(drop=True)

results

,dataset,filter,reads_pairs_unfiltered,reads_pairs_filtered,reads_kept_pct,scaffolds_unfiltered,scaffolds_filtered,bp_unfiltered,bp_filtered,seqs_unfiltered,seqs_filtered,nodes_unfiltered,nodes_filtered,edges_unfiltered,edges_filtered,avg_base_cov_unfiltered,avg_base_cov_filtered,run_root
0,10K_ext10_mixed,cnn,10000,7734,77.34,1,1,16612,16612,1,1,1,1,0,0,143.8,124.3,/Users/yvonnelin/Desktop/mitochime/data/assemb...
1,10K_ext10_mixed,gb,10000,9547,95.47,1,1,16612,16612,1,1,1,1,0,0,143.8,143.1,/Users/yvonnelin/Desktop/mitochime/data/assemb...
2,10K_ext10_mixed,trf,10000,8189,81.89,1,1,16612,16612,1,1,1,1,0,0,143.8,130.9,/Users/yvonnelin/Desktop/mitochime/data/assemb...
3,10K_ext20_mixed,cnn,10000,7059,70.59,1,1,16669,16615,1,1,1,1,0,0,134.7,111.2,/Users/yvonnelin/Desktop/mitochime/data/assemb...
4,10K_ext20_mixed,gb,10000,9126,91.26,1,1,16669,16597,1,1,1,1,0,0,134.7,134.0,/Users/yvonnelin/Desktop/mitochime/data/assemb...
5,10K_ext20_mixed,trf,10000,7561,75.61,1,1,16669,16615,1,1,1,1,0,0,134.7,117.2,/Users/yvonnelin/Desktop/mitochime/data/assemb...
6,10K_ext50_mixed,cnn,10000,4857,48.57,2,1,16680,16543,2,1,3,1,4,0,114.9,70.6,/Users/yvonnelin/Desktop/mitochime/data/assemb...
7,10K_ext50_mixed,gb,10000,7975,79.75,2,2,16680,16763,2,2,3,2,4,0,114.9,109.3,/Users/yvonnelin/Desktop/mitochime/data/assemb...
8,10K_ext50_mixed,trf,10000,5648,56.48,2,1,16680,16623,2,1,3,1,4,0,114.9,77.4,/Users/yvonnelin/Desktop/mitochime/data/assemb...
9,10K_ext5_mixed,cnn,10000,8114,81.14,1,1,16610,16610,1,1,1,1,0,0,149.2,131.6,/Users/yvonnelin/Desktop/mitochime/data/assemb...
